<a href="https://colab.research.google.com/github/fadeeva/Bayesian_Statistics_and_Quantitative_Finance/blob/main/01_Bayesian_estimation_of_volatility_with_unknown_drift/Bayesian_estimation_of_volatility_with_unknown_drift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd

import datetime as dt
import yfinance as yf

import arviz as az
import pymc as pm

import matplotlib.pyplot as plt
plt.style.use('ggplot')

# Bayesian estimation of volatility with unknown drift

## Data

In [12]:
ticker = '^GSPC'
start = dt.datetime(2023, 1, 1)
interval = '1d'
data = yf.download(ticker, start=start, interval=interval, auto_adjust=True)
data.to_csv('data.csv')

[*********************100%***********************]  1 of 1 completed


In [27]:
df = pd.read_csv('data.csv', parse_dates=True, header=[0,1], index_col=0)
df.head()

Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2023-01-03,3824.139893,3878.459961,3794.330078,3853.290039,3959140000
2023-01-04,3852.969971,3873.159912,3815.770020,3840.360107,4414080000
2023-01-05,3808.100098,3839.739990,3802.419922,3839.739990,3893450000
2023-01-06,3895.080078,3906.189941,3809.560059,3823.370117,3923560000
2023-01-09,3892.090088,3950.570068,3890.419922,3910.820068,4311770000


In [23]:
df.shape

(833, 5)

In [32]:
df['log_return'] = np.log(df['Close'].div(df['Close'].shift(1)))
df.dropna(axis=0, inplace=True)
df.head()

Price,Close,High,Low,Open,Volume,log_return
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC,
Date,,,,,,
2023-01-04,3852.969971,3873.159912,3815.770020,3840.360107,4414080000,0.007511
2023-01-05,3808.100098,3839.739990,3802.419922,3839.739990,3893450000,-0.011714
2023-01-06,3895.080078,3906.189941,3809.560059,3823.370117,3923560000,0.022584
2023-01-09,3892.090088,3950.570068,3890.419922,3910.820068,4311770000,-0.000768
2023-01-10,3919.250000,3919.830078,3877.290039,3888.570068,3851030000,0.006954


In [33]:
df.shape

(832, 6)

## Classical volatility assessment

In [34]:
df['log_return'].std()

0.009414373040479393